In [1]:
# Code from https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/15-frameworks.md

# The handwritten agent loop from the previous lesson is educational but repetitive. Every time you build a new agent, 
# you'd write the same while-loop, the same function-call handling, the same message management.

# ToyAIKit wraps this pattern so you can focus on tools, prompts, and behavior. We built it together in a DataTalks.Club workshop a while back. 
# It does the same thing as our handwritten loop with less boilerplate. 
# If you open its runners code, you'll find the same while True loop we wrote by hand.

# I use it here on purpose, because I don't want to pick a winner among the production frameworks. 
# ToyAIKit is small and easy to read, so when something breaks you can see exactly what happened. 
# That makes it handy for developing and debugging locally before you go to production.

# One caveat. ToyAIKit is a teaching and experimentation library, and it is NOT meant for production use. 
# We use it because it's minimal and you can see what it does.

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

# Load the data and build the search index - using python files, ??rewriting database??:

from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)


In [2]:
# Letting ToyAIKit generate the schema
# Writing that schema by hand is annoying, and we don't want to do it for every function. So we don't have to.

# If we add a type hint and a docstring to search, ToyAIKit reads them and derives the schema for us:

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )
    

In [3]:
# Registering the tool
# We register our search function along with the schema from earlier lessons:

agent_tools = Tools()
agent_tools.add_tool(search)

In [4]:
# You can look at what ToyAIKit produced.

agent_tools.get_tools()

# The output is the same JSON schema we hand-wrote in the function calling lesson. 
# ToyAIKit generated it from the docstring and the type hint.

# Every modern agent framework does this same trick. It reads a typed Python function with a 
# docstring and builds the schema from it. The OpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this way. 
# You write the tool and the framework figures out how to describe it.


[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [5]:
# Restricting off-topic questions - manual way via system prompt

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [6]:
# The chat interface and runner
# Create the chat interface and a callback, then build the runner:

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)
# The chat_interface handles display in the notebook. The callback renders model messages and tool calls as they happen. 
# The runner runs the agent loop, the same while True we wrote by hand. It sends messages, executes function calls, 
# adds tool outputs back, and repeats until the model is done.


In [7]:
# Running one prompt
# Run a single prompt:

result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

# We used the typo "Olama" on purpose. The agent searches and gets poor results, then retries with "Ollama". 
# The recovery is the same as the handwritten loop. The notebook output is nicer to watch. 
# Each tool call and message renders inline, so you can look at every search result.
    

-> Response received


-> Response received


-> Response received


In [8]:
# Cost and tokens
# Look at what the call cost:

result.cost

# Useful while developing - especially with multi-turn agents where one prompt can trigger several model calls. 
# The handwritten loop made you compute this by hand. The framework keeps a running total for you.


CostInfo(input_cost=Decimal('0.00268875'), output_cost=Decimal('0.0012915'), total_cost=Decimal('0.00398025'))